# Solution to p23

The first spatial violation occurs before concatenation: decoder height 7 does not equal skip height 8. Resizing the ground-truth mask to 7x8 neither repairs that mismatch nor preserves the required label/output contract; bilinear interpolation can also invent non-class mask values.

An exact architectural repair is to make the one-up decoder map a `4x4` bottleneck to `8x8` with `ConvTranspose2d(8,4,kernel_size=2,stride=2)` (and ensure the down path supplies that `4x4` bottleneck). Then concatenate decoder `(4,4,8,8)` with skip `(4,4,8,8)` on dim 1, fuse eight channels, and use a two-channel head to produce raw logits `(4,2,8,8)`. Keep the original masks `(4,8,8)` with dtype `torch.int64`; never bilinear-resize labels.

There are three connected leakage edges: `segmentation-heldout-00 -> fitted normalization -> training inputs`, `segmentation-heldout-00 -> fitted foreground threshold -> model/evaluation decisions`, and the epoch-4 held-out Dice can feed early stopping, selection, or further tuning back into training. Fit normalization only on the four immutable train IDs. Select any foreground threshold by a fixed train-only split or train-only cross-validation, then freeze it. Optimizer/backward steps use only those four train IDs. Keep both held-out IDs immutable and untouched until one final evaluation after architecture, normalization, threshold, and stopping rule are frozen. Compute hard foreground Dice with `eps=1e-7`.

### Answer check

In [ ]:
import torch
from torch import nn

bottleneck=torch.zeros(4,8,4,4)
skip=torch.zeros(4,4,8,8)
decoder=nn.ConvTranspose2d(8,4,kernel_size=2,stride=2)(bottleneck)
combined=torch.cat((decoder,skip),dim=1)
logits=nn.Conv2d(8,2,kernel_size=1)(combined)
masks=torch.zeros(4,8,8,dtype=torch.int64)
eps=1e-7
assert decoder.shape==(4,4,8,8)
assert combined.shape==(4,8,8,8)
assert logits.shape==(4,2,8,8)
assert masks.dtype==torch.int64 and eps==1e-7
train_ids={f"segmentation-train-{i:02d}" for i in range(4)}
heldout_ids={f"segmentation-heldout-{i:02d}" for i in range(2)}
assert train_ids.isdisjoint(heldout_ids)
